# Full Synteny Pipeline (Python version)

In [ ]:

# synteny_pipeline.py

# Step 1: Extract proteins from GFF and FASTA using gffread
# Run in terminal:
# gffread nig_chr7.gff -g nig_chr7.fa -y nig_chr7.protein.fa
# gffread pur_chr7.gff -g pur_chr7.fa -y pur_chr7.protein.fa

# Step 2: Clean protein files using awk
# awk '/^>/ {h=$0; next} /^[ACDEFGHIKLMNPQRSTVWY]+$/ {print h; print; h=""}' input.fa > cleaned.fa

# Step 3: Run BLASTP
# makeblastdb -in pur_chr7.cleaned.fa -dbtype prot
# blastp -query nig_chr7.cleaned.fa -db pur_chr7.cleaned.fa -evalue 1e-5 -outfmt 6 -num_threads 8 -out nig_vs_pur.blast

import pandas as pd
import matplotlib.pyplot as plt

# Step 4: Load gene coordinates from GFF-derived tables
nig_genes = pd.read_csv("nig_chr7_genes.tsv", sep="\t")
pur_genes = pd.read_csv("pur_chr7_genes.tsv", sep="\t")

# Step 5: Load and clean BLAST file
blast = pd.read_csv("nig_vs_pur.blast", sep="\t", header=None)
blast = blast.iloc[:, :2]
blast.columns = ["gene_nig", "gene_pur"]
blast["gene_nig"] = blast["gene_nig"].str.replace(r"\.\d+\.v5\.1", ".v5.1", regex=True)
blast["gene_pur"] = blast["gene_pur"].str.replace(r"\.\d+\.v5\.1", ".v5.1", regex=True)
blast = blast.drop_duplicates(subset=["gene_nig", "gene_pur"])

# Step 6: Merge with coordinates
merged = blast.merge(nig_genes, left_on="gene_nig", right_on="gene")
merged = merged.merge(pur_genes, left_on="gene_pur", right_on="gene", suffixes=("_nig", "_pur"))

# Step 7: Prepare midpoint coordinates and filter for regions
merged["mid_nig"] = (merged["start_nig"] + merged["end_nig"]) / 2 / 1e6
merged["mid_pur"] = (merged["start_pur"] + merged["end_pur"]) / 2 / 1e6
focus = merged[((merged["mid_nig"] >= 1) & (merged["mid_nig"] <= 2)) |
               ((merged["mid_nig"] >= 14.78) & (merged["mid_nig"] <= 16))].copy()

# Step 8: Filter for full-copy matches
focus["gene_core_nig"] = focus["gene_nig"].str.extract(r"(Sapur\.\d+G\d+)\.v5\.1")[0]
focus["gene_core_pur"] = focus["gene_pur"].str.extract(r"(Sapur\.\d+G\d+)\.v5\.1")[0]
focus = focus[focus["gene_core_nig"] == focus["gene_core_pur"]]

# Step 9: Plot final stacked synteny
fig, ax = plt.subplots(figsize=(14, 6))
ax.hlines(1, 0, merged["mid_nig"].max(), color="steelblue", linewidth=3, label="nig_chr7")
ax.hlines(0, 0, merged["mid_pur"].max(), color="indianred", linewidth=3, label="pur_chr7")
for _, row in focus.iterrows():
    ax.plot([row["mid_nig"], row["mid_pur"]], [1, 0], color="cornflowerblue", alpha=0.9, linewidth=1)
ax.set_xlabel("Genomic Position (Mb)")
ax.set_yticks([0, 1])
ax.set_yticklabels(["pur_chr7", "nig_chr7"])
ax.set_title("Stacked Synteny Plot (Full-Copy Genes: 1–2 Mb & 14.78–16 Mb of nig_chr7)")
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()
